# XLCoST — C

**Runtime → GPU (L4)**, then **Runtime → Run all**.

C is the smallest and most damaged language in XLCoST, and this notebook is
built to make that visible rather than to hide it. Read cell 3's verdict
before you trust anything cell 5 prints.

All figures below were produced by actually running cells 3a–3c on the corpus,
not estimated:

| | C | Python (for scale) |
|---|---|---|
| programs pooled from all three splits | 523 | 9,851 |
| programs lost to detokenization damage | 54 (10.3%) | 0.9% |
| occurrences after extraction | **345** | 16,671 |
| U+2581 contamination | **69.8%** | 0% |
| classes surviving `--min-class-count 20` | **3** | 4 |
| smallest test class | **2** | 9 |
| macro-F1 per one test occurrence (ρ) | **0.1145** | 0.0154 |

Three consequences, none of which the run itself will announce:

1. **C is a 3-class task.** `assignment` (8 occurrences) and `indexing_use` (5)
   fall below the class gate and are dropped, where the other four languages
   run 4-class. C macro-F1 is therefore **not comparable** to the Java,
   JavaScript, PHP, or Python tables.
2. **All three XLCoST splits are pooled** into one file. C/train alone yields
   286 occurrences and leaves only *one* class above the gate, which is not a
   classification task at all. Pooling is what makes C runnable; the 70/10/20
   fold used downstream is the protocol's own problem-hash split and is
   unaffected by it. Pooling dedupes by `problem_id` (532 rows → 523 programs).
3. **ρ = 0.1145 macro-F1 per test occurrence**, because `loop_use` has 2
   instances in the test fold. The largest effect anywhere in this study, the
   index role's −0.272, is **2.4 occurrences** at that scale; in Python it is
   17.7. Any difference smaller than about a tenth of a macro-F1 point is
   unmeasurable here, which includes every probe-vs-baseline margin observed
   in the other four languages.

C also has no separate extractor: it is registered to the C++ one, because C
is very nearly a subset and `tree-sitter-cpp` parses the XLCoST C corpus
better than `tree-sitter-c` does (381/421 against 363/421).


In [ ]:
# 1 - setup: clone/pull main, deps, token, restore prior work from Drive
# Reproduction/audit runs: set PIN_COMMIT to a full SHA to run vetted code
# instead of the moving branch. Use a READ-only HF token in Colab secrets.
PIN_COMMIT = ""
import os, pathlib
REPO = "/content/mech-interp"
if not pathlib.Path(REPO).exists():
    !git clone -q -b main https://github.com/nolanlwin/mech-interp.git {REPO}
%cd {REPO}
!git fetch -q origin
_old = !git rev-parse HEAD
if PIN_COMMIT:
    !git checkout -q {PIN_COMMIT}
else:
    !git checkout -q main 2>/dev/null || git checkout -q -b main origin/main
    !git pull -q
_new = !git rev-parse HEAD
if _old[0] != _new[0]:
    print("=" * 70)
    print("CODE CHANGED since this runtime last ran - review before trusting")
    print("the run with your HF token / Drive. New commits:")
    !git log --oneline {_old[0]}..{_new[0]}
    print("=" * 70)
!git log --oneline -1
# tree-sitter-cpp is what parses C here; without it cell 3 cannot extract.
!pip install -q transformers==5.8.0 tree_sitter "tree-sitter-cpp>=0.23.4" \
  "tree-sitter-java>=0.23.5" "tree-sitter-go>=0.25.0" \
  "tree-sitter-javascript>=0.25.0" "tree-sitter-php>=0.24.1" "tree-sitter-ruby>=0.23.1" \
  scikit-learn scipy huggingface_hub
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass
from google.colab import drive
drive.mount("/content/drive")
DEST = "/content/drive/MyDrive/mech-interp/xlcost"
!mkdir -p outputs/activations_xlcost outputs/probe_results outputs/xlcost_occ data/xlcost
!cp -rn {DEST}/stores/* outputs/activations_xlcost/ 2>/dev/null || true
!cp -n {DEST}/probe_results/* outputs/probe_results/ 2>/dev/null || true
!cp -n {DEST}/xlcost_occ/* outputs/xlcost_occ/ 2>/dev/null || true
!cp -n {DEST}/data_xlcost/* data/xlcost/ 2>/dev/null || true
print("setup complete")


In [ ]:
# 2 - CONFIG
LANGUAGE = "C"
SPLIT = "all"              # C pools train+valid+test; see cell 3
MODELS = [
    "Qwen/Qwen2.5-Coder-1.5B",
    "Qwen/Qwen2.5-1.5B",      # base-vs-Coder contrast at identical scale
    "bigcode/starcoder2-7b",  # different family, tokenizer, pretraining corpus
]

slug = LANGUAGE.lower().replace("++", "pp").replace("#", "sharp")
mslug = lambda m: m.split("/")[-1].lower().replace(".", "").replace("-", "")
RENAMING = False           # renamer v1 is Python-only
print(f"{LANGUAGE}/{SPLIT} | {len(MODELS)} model(s) | renaming: {RENAMING}")


In [ ]:
# 3 - corpus, pooling, extraction, and the FEASIBILITY VERDICT.
#     CPU only, ~1 min. Read the verdict before spending GPU time.
import json, subprocess, collections, hashlib
from pathlib import Path

# 3a. Build each XLCoST split, then pool. C/train alone leaves ONE class above
#     --min-class-count, which is not a classification task; pooling is what
#     makes C runnable. The 70/10/20 fold used downstream is the protocol's own
#     problem-hash split and does not come from these XLCoST split labels.
parts = []
for s in ("train", "valid", "test"):
    p = Path(f"data/xlcost/{slug}_{s}.jsonl")
    if not p.exists() or p.stat().st_size == 0:
        !python scripts/xlcost_data.py build --language "{LANGUAGE}" --split {s} --out-dir data/xlcost
    parts.append(p)

CANON = Path(f"data/xlcost/{slug}_{SPLIT}.jsonl")
seen, pooled = set(), []
for p in parts:
    for line in p.read_text().splitlines():
        if not line.strip():
            continue
        r = json.loads(line)
        if r["problem_id"] in seen:      # a problem can appear in >1 split
            continue
        seen.add(r["problem_id"])
        pooled.append(line)
CANON.write_text("\n".join(pooled) + "\n")
print(f"pooled corpus: {len(pooled)} programs -> {CANON}")

# 3b. Occurrences
OCC = Path(f"outputs/xlcost_occ/{slug}_{SPLIT}.jsonl")
if not OCC.exists() or OCC.stat().st_size == 0:
    !python scripts/xlcost_occurrences.py extract --input {CANON} --output {OCC}

# 3c. The verdict: class counts, the gate, and the measurement resolution.
rows = [json.loads(l) for l in OCC.read_text().splitlines() if l.strip()]
counts = collections.Counter(r["occurrence_type"] for r in rows)
MIN_CLASS = 20
dropped = {k: v for k, v in counts.items() if v < MIN_CLASS}
kept = [k for k in counts if counts[k] >= MIN_CLASS]

def fold(pid, seed=0):
    h = int(hashlib.sha1(f"{pid}:{seed}".encode()).hexdigest(), 16) % 100
    return "train" if h < 70 else ("valid" if h < 80 else "test")

test = collections.Counter(
    r["occurrence_type"] for r in rows
    if fold(str(r["problem_id"])) == "test" and r["occurrence_type"] in kept
)

print("\n" + "=" * 68)
print(f"FEASIBILITY VERDICT - {LANGUAGE}")
print("=" * 68)
print(f"occurrences        : {len(rows)} from {len({r['problem_id'] for r in rows})} problems")
print(f"class counts       : {dict(counts.most_common())}")
print(f"dropped (<{MIN_CLASS})     : {dropped or 'none'}")
print(f"task               : {len(kept)}-class ({', '.join(sorted(kept))})")
print(f"test fold          : {sum(test.values())} occurrences {dict(test)}")

RHO = None
if len(test) > 1:
    import numpy as np
    from sklearn.metrics import f1_score
    y = np.array([c for c, n in test.items() for _ in range(n)])
    small = min(test, key=test.get); big = max(test, key=test.get)
    yp = y.copy(); yp[np.where(y == small)[0][0]] = big
    labels = sorted(test)
    RHO = (f1_score(y, y, average="macro", labels=labels, zero_division=0)
           - f1_score(y, yp, average="macro", labels=labels, zero_division=0))
    print(f"smallest test class: {small} n={test[small]}")
    print(f"RESOLUTION (rho)   : {RHO:.4f} macro-F1 per ONE test occurrence")
    print("-" * 68)
    print("Any probe-vs-baseline margin below rho is noise, not a result.")
    print(f"For scale: the index role's -0.272 is {0.272/RHO:.1f} occurrences here;")
    print("Python's rho is 0.0154, so the same effect is 17.7 occurrences there.")
    if len(kept) != 4:
        print(f"\nNOTE: this is a {len(kept)}-class task. Java/JavaScript/PHP/Python")
        print("run 4-class, so these macro-F1 values are NOT comparable to them.")
else:
    print("\nSTOP: fewer than two classes survive the gate. A probe cannot run.")
print("=" * 68)


In [ ]:
# 4 - the sweep: probe + baselines per model. GPU.
#     Checkpoints to Drive after each model so a disconnect cannot lose one.
for m in MODELS:
    print(f"\n############ {LANGUAGE} - {m} ############")
    !bash scripts/run_language.sh {LANGUAGE} {m} {SPLIT}
    print(f"checkpoint: saving {LANGUAGE}/{m} to Drive")
    !mkdir -p {DEST}/probe_results {DEST}/stores {DEST}/xlcost_occ {DEST}/data_xlcost
    !cp -r outputs/probe_results/{slug}_{SPLIT}_* {DEST}/probe_results/ 2>/dev/null || true
    !cp -r outputs/activations_xlcost/{slug}_{SPLIT}_* {DEST}/stores/ 2>/dev/null || true


In [ ]:
# 5 - results table. Every number is printed against rho from cell 3, so a
#     margin smaller than one test occurrence cannot be read as a difference.
import glob, json

def best_baseline(path):
    try:
        return json.load(open(path)).get("strongest_baseline_macro_f1")
    except Exception:
        return None

print(f"=== {LANGUAGE}/{SPLIT} ===")
print(f"{'model':<22}{'C0 F1':>8}{'select.':>9}{'best base':>11}{'probe-base':>12}{'in rho units':>14}")
for m in MODELS:
    ms = mslug(m)
    pr = f"outputs/probe_results/{slug}_{SPLIT}_{ms}_problem.json"
    bs = f"outputs/probe_results/{slug}_{SPLIT}_{ms}_baselines_capped.json"
    try:
        d = json.load(open(pr))
    except Exception:
        print(f"{ms:<22}{'(missing)':>8}")
        continue
    f1 = d.get("macro_f1_mean") or d.get("macro_f1")
    sel = d.get("control_selectivity")
    base = best_baseline(bs)
    delta = (f1 - base) if (f1 is not None and base is not None) else None
    units = f"{delta / RHO:+.1f}" if (delta is not None and RHO) else "n/a"
    print(f"{ms:<22}{f1 if f1 is None else round(f1,4):>8}"
          f"{sel if sel is None else round(sel,4):>9}"
          f"{base if base is None else round(base,4):>11}"
          f"{delta if delta is None else round(delta,4):>12}{units:>14}")
    print(f"{'':<22}producing commit: {d.get('git_commit','?')}")

if RHO:
    print(f"\nrho = {RHO:.4f} macro-F1 per test occurrence.")
    print("A margin under +/-1.0 rho units is below what this sample can resolve.")


In [ ]:
# 6 - save this language's artifacts to Drive (language-scoped paths, so this
#     is safe to run while other languages' sessions are running).
!mkdir -p {DEST}/probe_results {DEST}/stores {DEST}/xlcost_occ {DEST}/data_xlcost
!cp -r outputs/probe_results/{slug}_{SPLIT}_* {DEST}/probe_results/ 2>/dev/null || true
!cp -r outputs/activations_xlcost/{slug}_{SPLIT}_* {DEST}/stores/ 2>/dev/null || true
!cp outputs/xlcost_occ/{slug}_{SPLIT}.jsonl* {DEST}/xlcost_occ/ 2>/dev/null || true
!cp data/xlcost/{slug}_*.jsonl {DEST}/data_xlcost/ 2>/dev/null || true
print(f"{LANGUAGE} artifacts saved to {DEST}")
